# Python practice 5: functions, decorators and typing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-practice-05-functions-and-typing.ipynb)

14 exercises on parameters, closures, functools, decorators, `match` and type hints. Each exercise names the [reference entries](../learning/python-reference-05-functions-and-typing.ipynb) it practises.

Write each function in the empty cell under its exercise and compare your results with the sample calls. Open the solution cell when you are done or stuck. 7 easy, 6 medium, 1 hard.

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Keyword-only options](#exercise-1-keyword-only-options) | Easy | [Parameter kinds](../learning/python-reference-05-functions-and-typing.ipynb#parameter-kinds) |
| 2 | [Fix the shared default](#exercise-2-fix-the-shared-default) | Easy | [Default values are evaluated once](../learning/python-reference-05-functions-and-typing.ipynb#default-values-are-evaluated-once) |
| 3 | [Running totals with a closure](#exercise-3-running-totals-with-a-closure) | Easy | [Scope, closures and `nonlocal`](../learning/python-reference-05-functions-and-typing.ipynb#scope-closures-and-nonlocal) |
| 4 | [Bind loop variables correctly](#exercise-4-bind-loop-variables-correctly) | Medium | [Scope, closures and `nonlocal`](../learning/python-reference-05-functions-and-typing.ipynb#scope-closures-and-nonlocal) |
| 5 | [Sort records by several keys](#exercise-5-sort-records-by-several-keys) | Easy | [`lambda`, `key=` functions and `operator`](../learning/python-reference-05-functions-and-typing.ipynb#lambda-key-functions-and-operator) |
| 6 | [Compose a pipeline of functions](#exercise-6-compose-a-pipeline-of-functions) | Medium | [`functools`: `partial`, `reduce`, `cache`, `lru_cache`](../learning/python-reference-05-functions-and-typing.ipynb#functools-partial-reduce-cache-lru_cache) |
| 7 | [Memoised path count](#exercise-7-memoised-path-count) | Medium | [`functools`: `partial`, `reduce`, `cache`, `lru_cache`](../learning/python-reference-05-functions-and-typing.ipynb#functools-partial-reduce-cache-lru_cache) |
| 8 | [Type-based serialisation](#exercise-8-type-based-serialisation) | Medium | [`functools.singledispatch`](../learning/python-reference-05-functions-and-typing.ipynb#functoolssingledispatch) |
| 9 | [A call-counting decorator](#exercise-9-a-call-counting-decorator) | Easy | [Writing a decorator](../learning/python-reference-05-functions-and-typing.ipynb#writing-a-decorator) |
| 10 | [Retry decorator with arguments](#exercise-10-retry-decorator-with-arguments) | Hard | [Decorators with arguments](../learning/python-reference-05-functions-and-typing.ipynb#decorators-with-arguments) |
| 11 | [Parse commands with match](#exercise-11-parse-commands-with-match) | Medium | [Structural pattern matching (`match`)](../learning/python-reference-05-functions-and-typing.ipynb#structural-pattern-matching-match) |
| 12 | [Extract with the walrus operator](#exercise-12-extract-with-the-walrus-operator) | Easy | [Assignment expressions (`:=`)](../learning/python-reference-05-functions-and-typing.ipynb#assignment-expressions-) |
| 13 | [Generic first-or-default](#exercise-13-generic-first-or-default) | Easy | [Generics, `TypedDict` and `Protocol`](../learning/python-reference-05-functions-and-typing.ipynb#generics-typeddict-and-protocol) |
| 14 | [Validate a TypedDict record](#exercise-14-validate-a-typeddict-record) | Medium | [Generics, `TypedDict` and `Protocol`](../learning/python-reference-05-functions-and-typing.ipynb#generics-typeddict-and-protocol) |

*Last reviewed: November 2026, against Python 3.13.*

In [ ]:
# Imports used by the sample calls and the solutions.
import io
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from fractions import Fraction

## Exercise 1: Keyword-only options

**Difficulty:** Easy · about 5 min · **Practises:** [Parameter kinds](../learning/python-reference-05-functions-and-typing.ipynb#parameter-kinds)

*Pipeline steps are safer when flags must be named at the call site.*

Write `make_path(bucket, table, *, date=None, fmt="parquet")` returning `bucket/table/date=<date>/data.<fmt>`, leaving out the `date=` part when `date` is None. `date` and `fmt` must be keyword-only.

**Sample calls**

| call | expected |
| --- | --- |
| `make_path("s3://raw", "orders")` | `'s3://raw/orders/data.parquet'` |
| `make_path("s3://raw", "orders", date="2026-11-27")` | `'s3://raw/orders/date=2026-11-27/data.parquet'` |
| `make_path("b", "t", fmt="csv")` | `'b/t/data.csv'` |

In [ ]:
def make_path(bucket, table, *, date=None, fmt="parquet"):
    # your code here
    pass

In [ ]:
def make_path(bucket, table, *, date=None, fmt="parquet"):
    parts = [bucket, table]
    if date is not None:
        parts.append(f"date={date}")
    parts.append(f"data.{fmt}")
    return "/".join(parts)

## Exercise 2: Fix the shared default

**Difficulty:** Easy · about 5 min · **Practises:** [Default values are evaluated once](../learning/python-reference-05-functions-and-typing.ipynb#default-values-are-evaluated-once)

*The mutable-default bug shows up in interviews and in real code that accumulates across calls.*

Write `add_tag(record, tag, tags=None)` returning a new list of tags: the given `tags` (not modified) plus `tag` if missing. Repeated calls without `tags` must not share state. `record` is unused context; return only the list.

**Sample calls**

| call | expected |
| --- | --- |
| `add_tag({}, "pii")` | `['pii']` |
| `(add_tag({}, "a"), add_tag({}, "b"))` | `(['a'], ['b'])` |
| `add_tag({}, "b", ["a"])` | `['a', 'b']` |

In [ ]:
def add_tag(record, tag, tags=None):
    # your code here
    pass

In [ ]:
def add_tag(record, tag, tags=None):
    result = list(tags) if tags is not None else []
    if tag not in result:
        result.append(tag)
    return result

## Exercise 3: Running totals with a closure

**Difficulty:** Easy · about 5 min · **Practises:** [Scope, closures and `nonlocal`](../learning/python-reference-05-functions-and-typing.ipynb#scope-closures-and-nonlocal)

*Closures hold small bits of state without a class.*

Write `make_accumulator(start=0)` returning a function `add(x)` that adds `x` to a running total and returns the total so far.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda acc: [acc(1), acc(2), acc(3)])(make_accumulator())` | `[1, 3, 6]` |
| `make_accumulator(10)(5)` | `15` |
| `(lambda a, b: (a(1), b(100), a(1)))(make_accumulator(), make_accumulator())` | `(1, 100, 2)` |

In [ ]:
def make_accumulator(start=0):
    # your code here
    pass

In [ ]:
def make_accumulator(start=0):
    total = start
    def add(x):
        nonlocal total
        total += x
        return total
    return add

## Exercise 4: Bind loop variables correctly

**Difficulty:** Medium · about 10 min · **Practises:** [Scope, closures and `nonlocal`](../learning/python-reference-05-functions-and-typing.ipynb#scope-closures-and-nonlocal)

*Late binding in closures is a classic interview trick question.*

Write `make_multipliers(factors)` returning a list of one-argument functions where the i-th multiplies its input by `factors[i]`.

**Sample calls**

| call | expected |
| --- | --- |
| `[m(10) for m in make_multipliers([1, 2, 3])]` | `[10, 20, 30]` |
| `make_multipliers([])` | `[]` |
| `len(make_multipliers(range(4)))` | `4` |

In [ ]:
def make_multipliers(factors):
    # your code here
    pass

In [ ]:
def make_multipliers(factors):
    return [lambda x, f=f: x * f for f in factors]

## Exercise 5: Sort records by several keys

**Difficulty:** Easy · about 5 min · **Practises:** [`lambda`, `key=` functions and `operator`](../learning/python-reference-05-functions-and-typing.ipynb#lambda-key-functions-and-operator)

*Ranking reports by several columns with mixed directions.*

Write `rank(rows)` sorting dicts by `region` ascending, then `amount` descending, then `id` ascending. Return the list of ids.

**Sample calls**

| call | expected |
| --- | --- |
| `rank([{"id": 1, "region": "s", "amount": 5}, {"id": 2, "region": "n", "amount": 5}, {"id": 3, "region": "n", "amount": 9}])` | `[3, 2, 1]` |
| `rank([{"id": 9, "region": "n", "amount": 1}, {"id": 4, "region": "n", "amount": 1}])` | `[4, 9]` |
| `rank([])` | `[]` |

In [ ]:
def rank(rows):
    # your code here
    pass

In [ ]:
def rank(rows):
    return [r["id"] for r in sorted(rows, key=lambda r: (r["region"], -r["amount"], r["id"]))]

## Exercise 6: Compose a pipeline of functions

**Difficulty:** Medium · about 10 min · **Practises:** [`functools`: `partial`, `reduce`, `cache`, `lru_cache`](../learning/python-reference-05-functions-and-typing.ipynb#functools-partial-reduce-cache-lru_cache)

*Composing small cleaning steps into one transform.*

Write `compose(*funcs)` returning a function that applies `funcs` left to right (`compose(f, g)(x) == g(f(x))`). With no functions it returns its input unchanged.

**Sample calls**

| call | expected |
| --- | --- |
| `compose(str.strip, str.lower)("  HeLLo ")` | `'hello'` |
| `compose()("same")` | `'same'` |
| `compose(lambda x: x + 1, lambda x: x * 10)(1)` | `20` |

In [ ]:
def compose(*funcs):
    # your code here
    pass

In [ ]:
from functools import reduce

def compose(*funcs):
    return lambda x: reduce(lambda acc, f: f(acc), funcs, x)

## Exercise 7: Memoised path count

**Difficulty:** Medium · about 10 min · **Practises:** [`functools`: `partial`, `reduce`, `cache`, `lru_cache`](../learning/python-reference-05-functions-and-typing.ipynb#functools-partial-reduce-cache-lru_cache)

*Memoisation turns exponential recursion into polynomial time, a staple of DP interview questions.*

Write `grid_paths(rows, cols)` returning the number of right/down paths from the top-left to the bottom-right of a grid, using `functools.cache`. A 1x1 grid has 1 path; non-positive sizes have 0.

**Sample calls**

| call | expected |
| --- | --- |
| `grid_paths(3, 3)` | `6` |
| `grid_paths(3, 7)` | `28` |
| `grid_paths(1, 1)` | `1` |

In [ ]:
def grid_paths(rows, cols):
    # your code here
    pass

In [ ]:
from functools import cache

def grid_paths(rows, cols):
    @cache
    def paths(r, c):
        if r <= 0 or c <= 0:
            return 0
        if r == 1 or c == 1:
            return 1
        return paths(r - 1, c) + paths(r, c - 1)
    return paths(rows, cols)

## Exercise 8: Type-based serialisation

**Difficulty:** Medium · about 10 min · **Practises:** [`functools.singledispatch`](../learning/python-reference-05-functions-and-typing.ipynb#functoolssingledispatch)

*Writing JSON from Python values that `json` cannot handle natively.*

Write `to_plain(value)` converting `Decimal` to str, `date`/`datetime` to ISO strings, sets to sorted lists, and recursing into lists, tuples (as lists) and dicts. Other values pass through.

**Sample calls**

| call | expected |
| --- | --- |
| `to_plain(Decimal("1.10"))` | `'1.10'` |
| `to_plain(datetime(2026, 11, 27, 9, 30))` | `'2026-11-27T09:30:00'` |
| `to_plain(date(2026, 1, 2))` | `'2026-01-02'` |

In [ ]:
def to_plain(value):
    # your code here
    pass

In [ ]:
from datetime import date
from decimal import Decimal
from functools import singledispatch

@singledispatch
def to_plain(value):
    return value

@to_plain.register
def _(value: Decimal):
    return str(value)

@to_plain.register
def _(value: date):
    return value.isoformat()

@to_plain.register
def _(value: set):
    return sorted(to_plain(v) for v in value)

@to_plain.register(list)
@to_plain.register(tuple)
def _(value):
    return [to_plain(v) for v in value]

@to_plain.register
def _(value: dict):
    return {k: to_plain(v) for k, v in value.items()}

## Exercise 9: A call-counting decorator

**Difficulty:** Easy · about 5 min · **Practises:** [Writing a decorator](../learning/python-reference-05-functions-and-typing.ipynb#writing-a-decorator)

*Instrumenting functions without touching their bodies.*

Write a decorator `counted(func)` that returns a wrapper preserving `__name__` and `__doc__` and exposing `wrapper.calls`, the number of times it has been called.

**Sample calls**

| call | expected |
| --- | --- |
| `counted(lambda x, y=1: x + y)(2, y=3)` | `5` |
| `(lambda f: (f(1), f(2), f.calls)[2])(counted(abs))` | `2` |
| `counted(abs).calls` | `0` |

In [ ]:
def counted(func):
    # your code here
    pass

In [ ]:
import functools

def counted(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

## Exercise 10: Retry decorator with arguments

**Difficulty:** Hard · about 15 min · **Practises:** [Decorators with arguments](../learning/python-reference-05-functions-and-typing.ipynb#decorators-with-arguments)

*Retrying transient failures is one of the most common decorators in data pipelines.*

Write `retry(times, exceptions=(Exception,))`, a decorator factory. The wrapped function is called up to `times` times while it raises one of `exceptions`; other exceptions propagate immediately; after the last failed attempt the exception propagates. Expose the attempts used on the last call as `wrapper.attempts`.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda s: (lambda f: (f(), f.attempts))(retry(3)(lambda: s.pop(0)() if s else 'ok')))([lambda: 1 / 0, lambda: 1 / 0])` | `('ok', 3)` |
| `(lambda f: (f(2), f.attempts))(retry(3)(lambda x: x * 2))` | `(4, 1)` |
| `raises(lambda: retry(2)(lambda: 1 / 0)(), ZeroDivisionError)` | `True` |

In [ ]:
def retry(times, exceptions=(Exception,)):
    # your code here
    pass

In [ ]:
import functools

def retry(times, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                wrapper.attempts = attempt
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise
        wrapper.attempts = 0
        return wrapper
    return decorator

## Exercise 11: Parse commands with match

**Difficulty:** Medium · about 10 min · **Practises:** [Structural pattern matching (`match`)](../learning/python-reference-05-functions-and-typing.ipynb#structural-pattern-matching-match)

*Routing events by shape is cleaner with `match` than nested ifs.*

Write `route(event)` for dict events: `{"op": "insert", "row": r}` → `"insert <len(r)> cols"`; `{"op": "delete", "id": i}` with int `i` → `"delete <i>"`; `{"op": "update" or "upsert", ...}` → `"write <op>"`; a list of events → list of routes; anything else → `"ignore"`.

**Sample calls**

| call | expected |
| --- | --- |
| `route({"op": "insert", "row": {"a": 1, "b": 2}})` | `'insert 2 cols'` |
| `route({"op": "delete", "id": 7})` | `'delete 7'` |
| `route({"op": "delete", "id": "7"})` | `'ignore'` |

In [ ]:
def route(event):
    # your code here
    pass

In [ ]:
def route(event):
    match event:
        case {"op": "insert", "row": dict() as row}:
            return f"insert {len(row)} cols"
        case {"op": "delete", "id": int() as row_id}:
            return f"delete {row_id}"
        case {"op": "update" | "upsert" as op}:
            return f"write {op}"
        case list():
            return [route(e) for e in event]
        case _:
            return "ignore"

## Exercise 12: Extract with the walrus operator

**Difficulty:** Easy · about 5 min · **Practises:** [Assignment expressions (`:=`)](../learning/python-reference-05-functions-and-typing.ipynb#assignment-expressions-)

*Filter and transform in one pass without matching twice.*

Write `durations(lines)` returning the integer milliseconds from lines containing `took <n>ms`, in order.

**Sample calls**

| call | expected |
| --- | --- |
| `durations(["job a took 120ms", "start", "job b took 7ms ok"])` | `[120, 7]` |
| `durations(["no timing"])` | `[]` |
| `durations([])` | `[]` |

In [ ]:
def durations(lines):
    # your code here
    pass

In [ ]:
import re

def durations(lines):
    return [int(m.group(1)) for line in lines if (m := re.search(r"took (\d+)ms", line))]

## Exercise 13: Generic first-or-default

**Difficulty:** Easy · about 5 min · **Practises:** [Generics, `TypedDict` and `Protocol`](../learning/python-reference-05-functions-and-typing.ipynb#generics-typeddict-and-protocol)

*Typed helpers that keep the element type for the checker.*

Write `first_match[T](items, predicate, default)` (or with a TypeVar) returning the first item for which `predicate(item)` is true, else `default`. Annotate it with a type parameter.

**Sample calls**

| call | expected |
| --- | --- |
| `first_match([1, 4, 6], lambda x: x % 2 == 0, -1)` | `4` |
| `first_match([1, 3], lambda x: x > 5, None)` | `None` |
| `first_match([], bool, "none")` | `'none'` |

In [ ]:
def first_match(items, predicate, default):
    # your code here
    pass

In [ ]:
from collections.abc import Callable, Iterable

def first_match[T](items: Iterable[T], predicate: Callable[[T], bool], default: T) -> T:
    return next((item for item in items if predicate(item)), default)

## Exercise 14: Validate a TypedDict record

**Difficulty:** Medium · about 10 min · **Practises:** [Generics, `TypedDict` and `Protocol`](../learning/python-reference-05-functions-and-typing.ipynb#generics-typeddict-and-protocol)

*TypedDict documents a shape but checks nothing at runtime, so ingestion code validates it.*

Given `class Order(TypedDict)` with `order_id: int`, `status: str` and `amount: float`, write `validate(record)` returning a sorted list of problems: `missing <key>`, `extra <key>`, or `bad <key>` when the value's type does not match (an int is fine for `amount`; a bool is never an int). Use `get_type_hints(Order)`.

**Sample calls**

| call | expected |
| --- | --- |
| `validate({"order_id": 1, "status": "paid", "amount": 9.5})` | `[]` |
| `validate({"order_id": 1, "status": "paid", "amount": 9})` | `[]` |
| `validate({"order_id": 1})` | `['missing amount', 'missing status']` |

In [ ]:
def validate(record):
    # your code here
    pass

In [ ]:
from typing import TypedDict, get_type_hints

class Order(TypedDict):
    order_id: int
    status: str
    amount: float

def validate(record):
    hints = get_type_hints(Order)
    problems = [f"missing {k}" for k in hints if k not in record]
    problems += [f"extra {k}" for k in record if k not in hints]
    for key, value in record.items():
        expected = hints.get(key)
        if expected is None:
            continue
        allowed = (int, float) if expected is float else (expected,)
        if isinstance(value, bool) or not isinstance(value, allowed):
            problems.append(f"bad {key}")
    return sorted(problems)

**Next:** look anything up in [Python reference 5: functions, decorators and typing](../learning/python-reference-05-functions-and-typing.ipynb), or try the [Python challenges](python-challenges-01-arrays-and-hashing.ipynb) and [Python coding problems](python-coding-problems.ipynb).